# persiscope: a full tour

Turn embedded data into topological summaries and compare them pairwise.

This notebook walks the whole API:

1. [The main entry point: `compare()`](#1)
2. [Inside a `Representation`](#2)
3. [Step-by-step: `TopologicalTransformer`](#3)
4. [Swappable diagram transforms](#4)
5. [Scoring: distribution scores vs. curve metrics](#5)
6. [Permutation p-values](#6)
7. [Confidence bands](#7)
8. [Other inputs: distance matrices and prebuilt graphs](#8)
9. [Visualization](#9)
10. [Saving and loading](#10)

Install with `pip install "persiscope[viz,metrics,frame]"`.

In [ ]:
import numpy as np
import persiscope as ps

print("persiscope", ps.__version__)
rng = np.random.default_rng(0)

# Synthetic "model embeddings" with known topology:
#   blob_a, blob_b — two draws of the same single gaussian cloud (same topology)
#   clusters      — two well-separated clouds (different H0 structure)
#   ring          — points on a noisy circle (interesting H1 structure)
blob_a = rng.normal(0, 1, size=(60, 8))
blob_b = rng.normal(0, 1, size=(60, 8))
clusters = np.vstack([rng.normal(-6, 1, size=(30, 8)), rng.normal(6, 1, size=(30, 8))])

theta = rng.uniform(0, 2 * np.pi, size=60)
ring2d = np.column_stack([np.cos(theta), np.sin(theta)]) * 5 + rng.normal(0, 0.15, size=(60, 2))
ring = np.hstack([ring2d, rng.normal(0, 0.15, size=(60, 6))])  # embed the circle in 8-D

embeddings = [blob_a, blob_b, clusters]
labels = ["blob_a", "blob_b", "clusters"]

<a id="1"></a>
## 1. The main entry point: `compare()`

A list of embedding arrays goes in and an all-pairs comparison comes out. Each input is fitted
into a `Representation` once, then every pair is scored, so the expensive topology is never
recomputed.

Distances inside each set are min-max normalized to `[0, 1]` by default (`normalize=True`), which is
what makes landscapes comparable across models with different embedding scales and dimensions.

In [ ]:
result = ps.compare(
    embeddings,
    method="energy",        # "energy" or any curve metric
    summary="landscape",    # "landscape" or "silhouette"
    homology_dim=0,
    n_bootstrap=50,
    labels=labels,
    random_state=0,
)

result.to_frame().round(4)   # the two blobs should be far closer to each other than to `clusters`

<a id="2"></a>
## 2. Inside a `Representation`

`compare()` keeps the fitted representations around so nothing needs refitting.

In [ ]:
rep = result.representations[0]
print(rep)
print()
print("persistence_diagram      ", rep.persistence_diagram.shape, " (birth, death) incl. essential class")
print("finite_diagram           ", rep.finite_diagram.shape, " essential class removed")
print("transformed_diagram      ", rep.transformed_diagram.shape, " after the diagram transform")
print("full_landscape           ", rep.full_landscape.shape, " whole-graph curve")
print("mean_landscape           ", rep.mean_landscape.shape, " bootstrap mean curve")
print("bootstrapped_landscapes  ", rep.bootstrapped_landscapes.shape, " one curve per subgraph draw")
print("landscape_band keys      ", sorted(rep.landscape_band))
print("normalization_factor     ", round(rep.normalization_factor, 4), " (raw max-min distance spread)")
print("params                   ", rep.params)

<a id="3"></a>
## 3. Step-by-step: `TopologicalTransformer`

`compare()` wraps two layers you can drive directly. The transformer takes embeddings through
the whole representation pipeline: weighted graph, Rips persistence, diagram transform, tent
functions, landscape and silhouette, subsampling bootstrap.

In [ ]:
tf = ps.TopologicalTransformer(
    homology_dim=0,           # 0 = components, 1 = loops
    n_bootstrap=50,           # subgraph draws
    subsample=0.8,            # fraction of nodes per draw
    silhouette_power=0.5,     # weight = |death - birth| ** power
    landscape_order=0,        # k-th landscape (0 = largest tent at each t)
    tenting_resolution=1000,  # samples along the t axis
    weight_method="euclidean",  # or "cosine", "manhattan"
    random_state=0,
)
rep_a = tf.fit_transform(blob_a, label="blob_a")
rep_b = tf.fit_transform(blob_b, label="blob_b")
rep_c = tf.fit_transform(clusters, label="clusters")
rep_a

### H1: loops

The ring embedding has a genuine 1-dimensional hole. Fitting with `homology_dim=1` summarizes loop
structure instead of component-merging structure.

In [ ]:
tf_h1 = ps.TopologicalTransformer(homology_dim=1, n_bootstrap=30, random_state=0)
rep_ring_h1 = tf_h1.fit_transform(ring, label="ring")
rep_blob_h1 = tf_h1.fit_transform(blob_a, label="blob_a")

# The ring's dominant H1 feature persists far longer than anything in the blob.
print("ring  max H1 persistence:", round(np.max(rep_ring_h1.finite_diagram[:, 1] - rep_ring_h1.finite_diagram[:, 0]), 3))
print("blob  max H1 persistence:", round(np.max(rep_blob_h1.finite_diagram[:, 1] - rep_blob_h1.finite_diagram[:, 0]), 3))

<a id="4"></a>
## 4. Swappable diagram transforms

Before tents are built, the diagram is mapped to new planar coordinates. The rotation angle
is the nonstandard part of the method, so it's exposed directly alongside `homology_dim`:

```python
ps.compare(embeddings, homology_dim=0, theta=-3 * np.pi / 8)   # the default
ps.TopologicalTransformer(theta=-np.pi / 4)                    # conventional rotation
```

If you need more than an angle, pass a full transform object instead: `RotateScale` (the
default), `H0Rotate`, `Identity`, or anything satisfying the `DiagramTransform` protocol
(a callable `(diagram, homology_dim) -> diagram`).

In [ ]:
from persiscope import transforms

default = transforms.RotateScale()                       # theta=-3π/8, alpha=√2/2
diagonal = transforms.RotateScale(theta=-np.pi / 4, alpha=np.sqrt(2) / 2)
h0_style = transforms.H0Rotate(angle=3 * np.pi / 8)

for t in (default, diagonal, h0_style, transforms.Identity()):
    rep_t = ps.TopologicalTransformer(diagram_transform=t, n_bootstrap=20, random_state=0).fit_transform(blob_a)
    print(f"{t!r:45s} -> peak landscape height {rep_t.mean_landscape[:, 1].max():.4f}")

A custom transform is just a callable, so a new coordinate scheme drops in without touching the pipeline:

In [ ]:
class BirthShift:
    """Toy custom transform: RotateScale, then shift births by a constant."""

    def __init__(self, shift=0.1):
        self.shift = shift
        self._base = transforms.RotateScale()

    def __call__(self, diagram, homology_dim=0):
        out = self._base(diagram, homology_dim).copy()
        if len(out):
            out[:, 0] += self.shift
        return out

rep_custom = ps.TopologicalTransformer(diagram_transform=BirthShift(), n_bootstrap=20, random_state=0).fit_transform(blob_a)
print("custom transform ran; landscape peak:", round(rep_custom.mean_landscape[:, 1].max(), 4))

<a id="5"></a>
## 5. Scoring: distribution scores vs. curve metrics

Two families share the `Scorer` interface. Larger always means more different:

| family | methods | compares |
| --- | --- | --- |
| distribution | `energy` | the two **bootstrap sets** of curves |
| curve | `euclidean`, `cosine`, `spectral`, `chi_squared`, `kl`, `js`, `wasserstein` (+ `dtw`, `frechet` with `[metrics]`) | the two **mean** curves |

In [ ]:
for method in ["energy", "euclidean", "js", "wasserstein"]:
    same = ps.Scorer(method=method, summary="landscape").score(rep_a, rep_b).score
    diff = ps.Scorer(method=method, summary="landscape").score(rep_a, rep_c).score
    print(f"{method:12s}  blob vs blob: {same:10.5f}   blob vs clusters: {diff:10.5f}")

In [ ]:
# Silhouettes are the persistence-weighted alternative to landscapes:
ps.Scorer(method="energy", summary="silhouette").score(rep_a, rep_c)

In [ ]:
# All-pairs matrix over already-fitted representations (what compare() uses internally):
sm = ps.score_matrix([rep_a, rep_b, rep_c], method="js", summary="landscape")
sm.to_frame().round(4)

<a id="6"></a>
## 6. Permutation p-values

Is an observed score bigger than chance? The one-sided permutation test pools the two bootstrap
sets, re-splits them at random, and recomputes the score. p-values are bounded below by
`1 / (n_permutations + 1)`, so they can never be exactly zero.

In [ ]:
scorer = ps.Scorer(method="energy", summary="landscape", run_pvalue=True, n_permutations=200, random_state=0)

res_same = scorer.score(rep_a, rep_b)
res_diff = scorer.score(rep_a, rep_c)
print("blob vs blob:    ", res_same)
print("blob vs clusters:", res_diff)

In [ ]:
# Or through compare() directly:
with_p = ps.compare(embeddings, method="energy", n_bootstrap=50, run_pvalue=True,
                    n_permutations=200, labels=labels, random_state=0)
with_p.pvalue_frame().round(3)

<a id="7"></a>
## 7. Confidence bands

Each mean summary carries a pointwise confidence band: the empirical `alpha/2` and
`1 - alpha/2` quantiles of the bootstrap curves at each t (default `band_alpha=0.05` for a
95% band), as in the original methodology. Bands are display-only: they shade the
plots but never enter scores or p-values.

In [ ]:
from persiscope.bands import pointwise_band

band = pointwise_band(list(rep_a.bootstrapped_landscapes), rep_a.mean_landscape, alpha=0.05)
print("mean half-width (95% band):", round(band["half_bound"].mean(), 5))

# A wider significance level -> narrower band:
band50 = pointwise_band(list(rep_a.bootstrapped_landscapes), rep_a.mean_landscape, alpha=0.5)
print("mean half-width (50% band):", round(band50["half_bound"].mean(), 5))

<a id="8"></a>
## 8. Other inputs: distance matrices and prebuilt graphs

If your notion of distance isn't a vector metric (edit distances, kernel dissimilarities,
graph distances), skip the embedding step entirely.

In [ ]:
from scipy.spatial.distance import cdist

# 8a. A precomputed square distance matrix per model:
dist_a = cdist(blob_a, blob_a)
dist_c = cdist(clusters, clusters)

rep_from_dist = tf.fit_transform(distance_matrix=dist_a, label="from_distances")
print(rep_from_dist)

# ...or a whole list of them through compare():
res_dist = ps.compare([dist_a, dist_c], input_kind="distance_matrix",
                      method="energy", n_bootstrap=30, labels=["a", "c"], random_state=0)
print("score:", round(res_dist.matrix[0, 1], 4))

In [ ]:
# 8b. A prebuilt networkx graph (weights = distances). Note: passed through untouched —
# you are responsible for its weight scale (persiscope does not re-normalize it).
import networkx as nx

g, norm = ps.build_graph(blob_a)          # the same builder the transformer uses
print(type(g).__name__, g.number_of_nodes(), "nodes; normalization_factor:", round(norm, 3))

rep_from_graph = tf.fit_transform(graph=g, label="from_graph")
print(rep_from_graph, "| normalization_factor:", rep_from_graph.normalization_factor)  # None: unknown for prebuilt graphs

<a id="9"></a>
## 9. Visualization (`[viz]` extra)

Every helper returns a figure object; you decide whether to `show()` or `savefig()`.

### The baseline report

`plot_baseline_report` condenses the whole analysis into one figure, comparing every model to a
**baseline**: overlaid mean landscapes and silhouettes with 95% confidence bands (top), the
energy statistic vs. the baseline per summary type with permutation-significance stars (middle),
and Wasserstein/JS distance strips annotated with stars (bottom).

In [ ]:
fig = ps.viz.plot_baseline_report(
    result.representations,
    baseline=0,                    # every model is compared to blob_a
    curve_metrics=("wasserstein", "js"),
    n_permutations=200,
    random_state=0,
)

In [ ]:
ps.viz.plot_persistence_diagram(rep_c);   # essential class drawn as a triangle at the top

In [ ]:
ps.viz.plot_landscape(rep_c)              # mean curve + confidence band shading
ps.viz.plot_silhouette(rep_c);

In [ ]:
# Compare curves across models on one axis:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 3.5))
for r in (rep_a, rep_b, rep_c):
    ax.plot(r.mean_landscape[:, 0], r.mean_landscape[:, 1], label=r.label)
ax.set_xlabel("t"); ax.set_ylabel("landscape"); ax.legend(); ax.set_title("Mean landscapes")
plt.show()

In [ ]:
fig = ps.viz.plot_score_heatmap(result)   # plotly; accepts a ComparisonResult, ScoreMatrix, or raw array
fig.show()

### The comparison report

`plot_comparison_report` condenses the whole analysis into one figure: mean summaries with
confidence bands on top, the energy-statistic permutation test per pair in the middle, and
annotated curve-distance heatmaps (score + p-value per cell) on the bottom.

In [ ]:
fig = ps.viz.plot_comparison_report(
    result.representations,
    summary="landscape",
    curve_metrics=("js", "wasserstein"),
    n_permutations=200,
    random_state=0,
)

<a id="10"></a>
## 10. Saving and loading

Representations are expensive to fit; `ps.save` / `ps.load` round-trip them through a compressed
`.npz` so downstream scoring and plotting never refit.

In [ ]:
import tempfile
from pathlib import Path

path = Path(tempfile.mkdtemp()) / "blob_a.npz"
ps.save(rep_a, path)
reloaded = ps.load(path)

print(reloaded)
print("round-trip exact:", np.allclose(reloaded.mean_landscape, rep_a.mean_landscape))
print("score(reloaded, rep_c):", round(ps.Scorer(method="energy").score(reloaded, rep_c).score, 4))

---

### Next steps

- Real embeddings: pass sentence, image, or model activations as `(n, d)` arrays to `ps.compare`.
  Per-set distance normalization makes models of different dimensionality comparable.
- H1 and beyond: `homology_dim=1` compares loop structure (slower, since the Rips complex grows fast).
- New coordinate schemes: implement the small `DiagramTransform` protocol and pass it to the
  transformer. Bootstrap, bands, and scoring all come along for free.